In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from botorch.utils.multi_objective import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated

In [ ]:
def get_pareto(Y):
    idx_pareto = is_non_dominated(torch.from_numpy(Y))
    idx_sort = np.argsort(Y[idx_pareto, 0])
    idx_combined = np.where(idx_pareto)[0][idx_sort]

    return Y[idx_combined]


def get_pareto_hv(Y, hypervolume_fun):
    Y_pareto = get_pareto(Y)
    hv = hypervolume_fun.compute(torch.from_numpy(Y))
    return Y_pareto, hv


def load_data(file_name):
    data = np.load(file_name)
    X = data["train_X"]
    Y = data["train_Y"]
    return X, Y

In [ ]:
colors = ["black", "blue", "orange", "red", "green", "pink"]
methods_rename = {
    "HacklGenerator_OneLambda": ("Same lambda", "black"),
    "HacklGenerator_SixLambdas": ("Six lambdas", "red"),
    "HacklGenerator_3BrokenLines": ("3 BrokenLines", "pink"),
}
roots = [f"../results/results{i}" for i in range(1, 6)]

max_ripple = 0.1
ref_point = [4.0, -max_ripple]
hypervolume_fun = Hypervolume(torch.tensor(ref_point))

labels = ["All designs", "Initial designs"]
methods = list(methods_rename.keys())

In [ ]:
X = {}
Y = {}
for use_constraints in [True, False]:
    for method in methods:
        for root in roots:
            key = method, use_constraints, root
            file_name = f"{root}/results_{method}_{use_constraints}.npz"
            X[key], Y[key] = load_data(file_name)

In [ ]:
for i, use_constraints in enumerate([True, False]):
    Y_all = np.empty((0, 2))
    for method in methods:
        for root in roots:
            key = method, use_constraints, root
            Y_all = np.vstack((Y_all, Y[key]))

    Y_pareto, hypervolume = get_pareto_hv(Y_all, hypervolume_fun)
    Y_pareto[:, 1] *= -100
    Y_all[:, 1] *= -100

    plt.scatter(Y_all[:, 0], Y_all[:, 1], color="gray", alpha=0.2, s=12, zorder=1, label=labels[0] if i == 0 else "")
    if i == 0:
        color = "black"
        label = f"Pareto front with constraints (HV={100 * hypervolume:.2f})"
    else:
        color = "red"
        label = f"Pareto front without constraints (HV={100 * hypervolume:.2f})"
    plt.scatter(Y_pareto[:, 0], Y_pareto[:, 1], color=color, s=14, label=label)
    plt.plot(Y_pareto[:, 0], Y_pareto[:, 1], color=color)
plt.scatter([ref_point[0]], [-100 * ref_point[1]], color="black", label="Reference point", marker="*")

plt.xlim([3.6, 4.6])
plt.ylim([0, 25])
plt.legend()
plt.grid()
ax = plt.gca()
ax.set_axisbelow(True)
plt.savefig("all_designs.png", dpi=600, bbox_inches="tight")

In [ ]:
plt.figure(figsize=(6, 4))
for i, use_constraints in enumerate([True, False]):
    ax = plt.subplot(1, 2, i + 1)
    for method in methods:
        Y_all = np.empty((0, 2))
        for root in roots:
            key = method, use_constraints, root
            Y_all = np.vstack((Y_all, Y[key]))

        Y_pareto = get_pareto(Y_all)
        Y_pareto[:, 1] *= -100

        label, color = methods_rename[method]
        plt.plot(Y_pareto[:, 0], Y_pareto[:, 1], label=label, color=color, linewidth=3)
    ax.axhline(10, color="black", linestyle=":", linewidth=2, alpha=0.7)

    if i == 0:
        plt.legend()
        plt.title("With constraints")
        plt.ylabel("RMS ripple [%]")
    else:
        plt.title("Without constraints")

    plt.xlabel("Mean torque [Nm]")
    plt.xlim([4.1, 4.6])
    plt.ylim([2, 20])
    plt.grid(alpha=0.35)
    ax.fill_between([4.1, 4.6], 0, 10, color="green", alpha=0.05)
plt.savefig("result.png", dpi=600, bbox_inches="tight")

In [ ]:
i = 0
x_labels = []
for use_constraints in [True, False]:
    for method in methods:
        for root in roots:
            key = method, use_constraints, root
            Y_pareto, hypervolume = get_pareto_hv(Y[key], hypervolume_fun)
            plt.scatter([i], [hypervolume])
        i += 1
        x_labels.append(f"{use_constraints}_{method}")
# plt.x(x_labels)
plt.xticks(range(len(x_labels)), x_labels, rotation=90)